# Lab: Verified Bank Research

Build a **grounded, self-verifying research agent** on Snowflake on synthetic data (run `app/setup_synthetic.sql`), or on free public call-report data (run `app/setup.sql`).

**You will build:** given a bank + peers, produce a one-page research note (scale, profitability, capital) using *only* governed FFIEC/FDIC call-report data, then run a **verifier** that strips any unsourced claim and flags hedged reports as *Needs review*.

| Step | What you build |
|---|---|
| 1 | Explore the governed data |
| 2 | Query the **semantic view** (Cortex Analyst) |
| 3 | Build a verified **fact pack** |
| 4 | **Grounded draft** under DATA RULES |
| 5 | **Verify** claims vs evidence |
| 6 | Persist the note |

**Prerequisite:** run `app/setup_synthetic.sql` (or `app/setup.sql`) first (creates `SNOWPACK_BANK_RESEARCH.CORE`: the fact view, the semantic view, and the `REPORTS` table). With `setup.sql` the data comes from the free `SNOWFLAKE_PUBLIC_DATA_FINANCIAL_PACKAGE` Marketplace listing; with `setup_synthetic.sql` it is generated in your account.

In [ ]:
# Parameters — edit these, then Run All
SCHEMA        = "SNOWPACK_BANK_RESEARCH.CORE"
MODEL         = "claude-sonnet-4-5"   # any Cortex model your account has, e.g. llama3.1-70b, mistral-large2
SUBJECT_BANK  = "Ombrel National Bank"   # synthetic data; pick any BANK_NAME on real data
PEER_BANKS    = [
    "Tarrowind Commercial Bank",
    "Vessick Federal Savings",
    "Kestrova Bank",
]
AS_OF_QUARTER = "2026-06-30"   # reporting quarter to analyze

import json
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print("Connected as", session.sql("select current_user(), current_role()").collect()[0])

## Step 1 — Explore the governed data

`BANK_QUARTERLY_METRICS` is one clean row per (bank, quarter). It is the agent's **only** allowed source of numbers — so the agent cannot invent figures.

In [ ]:
names = "', '".join([SUBJECT_BANK] + PEER_BANKS)
df = session.sql(f"""
    SELECT bank_name, state, quarter_end,
           total_assets, total_deposits, net_income_qtr,
           return_on_assets, return_on_equity, equity_to_assets
    FROM {SCHEMA}.BANK_QUARTERLY_METRICS
    WHERE bank_name IN ('{names}') AND quarter_end = '{AS_OF_QUARTER}'
    ORDER BY total_assets DESC
""").to_pandas()
df

## Step 2 — Cortex Analyst over the semantic view

`BANK_RESEARCH_SV` adds business meaning (synonyms, metric definitions) on top of the view so **Cortex Analyst** can answer natural-language questions. Below we query it directly with the `SEMANTIC_VIEW(...)` syntax; in Snowflake Intelligence / a Cortex Agent, Analyst writes this SQL for you from a plain-English question like *"top 5 banks by total assets in 2026"*.

In [ ]:
session.sql(f"""
    SELECT * FROM SEMANTIC_VIEW({SCHEMA}.BANK_RESEARCH_SV
        METRICS total_assets_usd, avg_return_on_assets
        DIMENSIONS bank_name, report_year)
    WHERE report_year = YEAR('{AS_OF_QUARTER}'::date)
    ORDER BY total_assets_usd DESC
    LIMIT 10
""").to_pandas()

## Step 3 — Build the verified fact pack

The fact pack is a small JSON bundle of **only** the verified rows the agent may cite — the subject bank and peers, latest quarter plus one year prior (for a trend). Precomputed, governed, copy-exactly.

In [ ]:
# Same logic as app/research.py, inlined so the notebook is self-contained.
def _rows(session, sql, params=None):
    return session.sql(sql, params=params or []).collect()


def build_fact_pack(session, names, quarter):
    """Reporting-quarter and year-ago rows for the chosen banks. The reporting-quarter rows also carry the
    year-over-year changes (percentage points) and the asset rank within the group, so the model copies
    them instead of calculating."""
    usd = lambda c: (f"IFF({c} < 0, '-', '') || '$' || CASE WHEN ABS({c}) >= 1e12 THEN TO_CHAR(ABS({c}) / 1e12, 'FM990.00') || 'T' "
                     f"WHEN ABS({c}) >= 1e9 THEN TO_CHAR(ABS({c}) / 1e9, 'FM9990.0') || 'B' ELSE TO_CHAR(ABS({c}) / 1e6, 'FM9990.0') || 'M' END")
    row = _rows(session, f"""
        WITH m AS (
          SELECT * FROM {SCHEMA}.BANK_QUARTERLY_METRICS
          WHERE ARRAY_CONTAINS(bank_name::VARIANT, PARSE_JSON(?))
            AND quarter_end IN (?::DATE, DATEADD(year, -1, ?::DATE))
        ), x AS (
          SELECT m.*,
                 LEAD(return_on_assets) OVER (PARTITION BY id_rssd ORDER BY quarter_end DESC) AS roa_prev,
                 LEAD(return_on_equity) OVER (PARTITION BY id_rssd ORDER BY quarter_end DESC) AS roe_prev,
                 LEAD(equity_to_assets) OVER (PARTITION BY id_rssd ORDER BY quarter_end DESC) AS eqta_prev,
                 RANK() OVER (PARTITION BY quarter_end ORDER BY total_assets DESC) AS asset_rank,
                 COUNT(*) OVER (PARTITION BY quarter_end) AS group_size
          FROM m
        )
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT_KEEP_NULL(
                 'bank', bank_name, 'peer_group', peer_group, 'quarter_end', quarter_end::STRING,
                 'total_assets_usd', total_assets, 'total_deposits_usd', total_deposits,
                 'net_income_qtr_usd', net_income_qtr,
                 'roa_fraction', return_on_assets, 'roe_fraction', return_on_equity,
                 'equity_to_assets_fraction', equity_to_assets,
                 'total_assets_display', {usd('total_assets')}, 'total_deposits_display', {usd('total_deposits')},
                 'net_income_qtr_display', {usd('net_income_qtr')},
                 'roa_display', TO_CHAR(return_on_assets * 100, 'FM990.00') || '%',
                 'roe_display', TO_CHAR(return_on_equity * 100, 'FM990.00') || '%',
                 'equity_to_assets_display', TO_CHAR(equity_to_assets * 100, 'FM990.00') || '%',
                 'asset_rank_in_group', asset_rank || ' of ' || group_size,
                 'roa_change_pp_vs_year_ago', IFF(roa_prev IS NULL, NULL, ROUND((return_on_assets - roa_prev) * 100, 2)),
                 'roa_change_display', IFF(roa_prev IS NULL, NULL, TO_CHAR((return_on_assets - roa_prev) * 100, 'FMS0.00') || ' pp'),
                 'roe_change_display', IFF(roe_prev IS NULL, NULL, TO_CHAR((return_on_equity - roe_prev) * 100, 'FMS0.00') || ' pp'),
                 'equity_to_assets_change_display', IFF(eqta_prev IS NULL, NULL, TO_CHAR((equity_to_assets - eqta_prev) * 100, 'FMS0.00') || ' pp'),
                 'roe_change_pp_vs_year_ago', IFF(roe_prev IS NULL, NULL, ROUND((return_on_equity - roe_prev) * 100, 2)),
                 'equity_to_assets_change_pp_vs_year_ago', IFF(eqta_prev IS NULL, NULL, ROUND((equity_to_assets - eqta_prev) * 100, 2))))
                 WITHIN GROUP (ORDER BY quarter_end DESC, total_assets DESC) AS pack
        FROM x""",
        [json.dumps(names), quarter, quarter])
    pack = json.loads(row[0][0])
    # year-ago rows only need the levels
    for r in pack:
        if r["quarter_end"] != quarter:
            for k in ("roa_change_pp_vs_year_ago", "roe_change_pp_vs_year_ago",
                      "equity_to_assets_change_pp_vs_year_ago", "roa_change_display", "roe_change_display",
                      "equity_to_assets_change_display"):
                r.pop(k, None)
    return pack


fact_pack = build_fact_pack(session, [SUBJECT_BANK] + PEER_BANKS, AS_OF_QUARTER)
print(json.dumps(fact_pack[0], indent=1))
print(len(fact_pack), "rows in the fact pack")


## Step 4 — Grounded draft (DATA RULES)

The system prompt is the heart of a *verified* agent. The **DATA RULES** forbid outside numbers, estimates, and hedging — the model may only use the fact pack.

In [ ]:
DATA_RULES = (
    "You are a bank research analyst. Write a concise peer research note on the SUBJECT bank versus its peers "
    "for the reporting quarter, using ONLY the FACTS JSON.\n"
    "DATA RULES (mandatory):\n"
    "1. Use ONLY numbers present in FACTS. Copy them exactly. Never estimate, extrapolate or add outside figures.\n"
    "2. Every figure you write must be copied verbatim from a *_display field (for example total_assets_display "
    "\"$151.7B\", roa_display \"0.46%\"). Never reformat, round or convert the raw *_usd or *_fraction values.\n"
    "3. Year-over-year changes: copy the *_change_display field without its sign, written like \"down 0.48 pp\" or "
    "\"up 0.02 pp\".\n"
    "4. No hedging words (likely, approximately, around, could, industry-leading) and no market or consensus claims.\n"
    "5. If a figure is missing, write n/d. Do not invent a row. Do not name any real bank or company.\n"
    "6. Do not calculate anything. Do not state ratios, multiples or relative changes (\"half\", \"twice\", "
    "\"x% decline\"). For size rankings copy asset_rank_in_group, written like \"3rd of 4 by total assets\".\n"
    "OUTPUT: markdown starting with '# ', then a one-line '**Takeaway.**' paragraph, a peer comparison table "
    "(Bank, Total assets, Deposits, ROA, ROE, Equity/assets) for the reporting quarter, then exactly 3 short bullets "
    "(one on scale, one on profitability with the year-ago change, one on capital). End with a '## Data coverage' "
    "line naming the source (BANK_QUARTERLY_METRICS, synthetic call-report data) and the quarters used."
)

prompt = f"{DATA_RULES}\n\nSUBJECT: {SUBJECT_BANK}\nREPORTING QUARTER: {AS_OF_QUARTER}\nFACTS: {json.dumps(fact_pack)}"
draft_md = json.loads(session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, prompt]).collect()[0][0])
print(draft_md)


## Step 5 — Verify the draft against the evidence

A second model pass fact-checks every claim against the fact pack and counts residual hedges. If anything is unsupported or hedged, the note is flagged **Needs review**.

In [ ]:
VERIFY_RULES = (
    "You are a fact-checker for bank research notes. Compare the DRAFT against the FACTS JSON, the only allowed "
    "source. Every figure in the draft must match a *_display value in FACTS for the same bank and quarter, "
    "character for character (a change written \"down 0.48 pp\" matches \"-0.48 pp\"). The display values are "
    "already correctly rounded: never re-derive them from the raw *_usd or *_fraction fields. Rankings and "
    "comparisons (largest, smallest, lowest, 2nd, above, below, trailing, middle of the range) must hold in FACTS. "
    "Hedge words are likely, approximately, around, could, industry-leading.\n"
    "Return one finding per factual claim you checked. reason first: one short sentence comparing the claim with "
    "the FACTS value. Then the verdict, SUPPORTED, UNSUPPORTED or HEDGED, which must agree with the reason. "
    "residual_hedges counts hedge words."
)

VERIFY_FORMAT = (
    "TYPE OBJECT(status STRING, residual_hedges NUMBER, "
    "findings ARRAY(OBJECT(claim STRING, reason STRING, verdict STRING)))"
)

# Structured output: AI_COMPLETE must return an object of exactly this shape.
verify_prompt = f"{VERIFY_RULES}\n\nFACTS: {json.dumps(fact_pack)}\n\nDRAFT:\n{draft_md}"
raw = session.sql(f"SELECT AI_COMPLETE(model => ?, prompt => ?, response_format => {VERIFY_FORMAT})",
                  params=[MODEL, verify_prompt]).collect()[0][0]
checked = json.loads(raw).get("findings") or []
rejected = [f for f in checked if str(f.get("verdict", "")).upper() != "SUPPORTED"]
status = "verified" if checked and not rejected else "needs_review"
edits = [f"{f['verdict']}: {f['claim']} ({f['reason']})" for f in rejected]
hedges = json.loads(raw).get("residual_hedges")
print(f"STATUS: {status.upper()}   claims checked={len(checked)}   rejected={len(rejected)}")
for e in edits:
    print(" -", e)


## Step 6 — Persist the note

Save the verified note to `REPORTS`, tagged with its status — the store a Streamlit gallery or a scheduled task would read from.

In [ ]:
session.sql(
    f"INSERT INTO {SCHEMA}.REPORTS (subject, draft_md, report_md, status, edits_applied, residual_hedges) "
    f"SELECT ?, ?, ?, ?, ?, ?",
    params=[SUBJECT_BANK, draft_md, draft_md, status, len(edits), hedges]
).collect()
session.sql(f"SELECT run_id, subject, status, edits_applied, residual_hedges, created_at FROM {SCHEMA}.REPORTS ORDER BY created_at DESC LIMIT 5").to_pandas()

## Extension A — Turn it into a real Cortex Agent

Steps 3–5 showed the *verified* pattern explicitly. To give a PM a conversational interface, wrap the semantic view in a **Cortex Agent** with a Cortex Analyst tool and the same grounding instructions. Run the SQL below in a SQL cell (or Snowsight worksheet):

```sql
CREATE OR REPLACE AGENT SNOWPACK_BANK_RESEARCH.CORE.BANK_RESEARCH_AGENT
  COMMENT = 'Verified bank research agent (Cortex Analyst over BANK_RESEARCH_SV)'
  PROFILE = '{"display_name": "Bank Research Agent", "color": "blue"}'
  FROM SPECIFICATION
  $$
  models:
    orchestration: auto
  orchestration:
    capabilities:
      analytical_search: true
  instructions:
    response: 'You are a buy-side bank analyst. Use ONLY figures returned by the Analyst tool; never invent numbers. ROA/ROE/equity ratios are fractions (0.015 = 1.5%). Express dollars in $B/$T. Be concise and cite the quarter.'
    orchestration: 'For any bank metric, ranking, or comparison, use the BankMetrics Analyst tool.'
    sample_questions:
      - question: 'Top 5 banks by total assets in 2026'
      - question: "What was Ombrel National Bank's ROA and ROE in Q2 2026?"
  tools:
    - tool_spec:
        type: cortex_analyst_text_to_sql
        name: BankMetrics
        description: 'Quarterly FFIEC/FDIC call-report metrics (assets, deposits, net income, ROA, ROE, capital, loans) per US bank.'
  tool_resources:
    BankMetrics:
      semantic_view: SNOWPACK_BANK_RESEARCH.CORE.BANK_RESEARCH_SV
  $$;
```

**Chat with it:** open Snowsight → **AI & ML → Agents → Bank Research Agent**. The agent runs with the role selected in the UI, so that role needs `USAGE` on the agent and on the semantic view's data. To let another role use it:

```sql
GRANT USAGE ON AGENT SNOWPACK_BANK_RESEARCH.CORE.BANK_RESEARCH_AGENT TO ROLE <role>;
```

> The agent reuses the **same** semantic view you built, so Cortex Analyst only returns governed numbers — the grounding guarantee carries over to the conversational experience.

## Recap & extensions

You built the full **verified research agent** loop: governed semantic view → verified fact pack → grounded draft under DATA RULES → verification gate → stored note. 

**Graduation extensions (try next):**
1. **Turn it into a real Cortex Agent** — done in *Extension A* above (`CREATE AGENT` over `BANK_RESEARCH_SV`); chat with it in Snowflake Intelligence.
2. **Add an unstructured tool** — build a `CORTEX SEARCH SERVICE` over `FINANCIAL_CFPB_COMPLAINT` narratives (scope it tightly) and let the agent cite consumer-complaint themes.
3. **Schedule it** — wrap Steps 3–6 in a stored proc and a `TASK` to run a watchlist weekly.
4. **Surface it** — a Streamlit card gallery reading `REPORTS`.

**Try breaking the verifier:** set `MODEL` to a weaker model, or edit the draft to add a made-up number, and watch Step 5 flag it as *Needs review*.